# **GAN Training**

### **Data loading**

In [ ]:
import tensorflow as tf
import numpy as np
import tensorflow_datasets as tfds

ds = tfds.load('qm9', split='train')
ds = tfds.as_dataframe(ds)

features_to_drop = ['InChI', 'InChI_relaxed', 'SMILES_relaxed', 'tag', 'frequencies', 
                    'charges', 'positions', 'Mulliken_charges', 'num_atoms', 'index']
ds = ds.drop(features_to_drop, axis=1)

ds_smiles = ds['SMILES'].astype('string')
ds_props = ds.drop(['SMILES'], axis=1)

2026-05-06 11:36:23.796864: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778060187.071489   16951 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13483 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4070 Ti SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9
2026-05-06 11:36:27.451450: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
2026-05-06 11:37:17.749761: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


### **Tokenizated data loading**

In [2]:
import json

with open('Transformer/token_to_id.json', 'r') as f:
    token_to_id = json.load(f)
with open('Transformer/id_to_token.json', 'r') as f:
    id_to_token = json.load(f)

### **Data Preprocessing**

In [ ]:
import joblib
from sklearn.model_selection import train_test_split
from utils_functions import smiles_to_selfies_tokens_ds, prepare_smiles_to_decoder_ds, get_props

# Get additional properties
add_props = get_props(ds_smiles)
ds_props = np.concatenate([ds_props, add_props], axis=1)

train_smiles, _, train_props, _ = train_test_split(ds_smiles, ds_props, test_size=0.2, random_state=42)
train_smiles, _, train_props, _ = train_test_split(train_smiles, train_props, test_size=0.2, random_state=42)

SEQ_LEN = 25    # Sequence length

# Encoding to selfies format
_, indieces_to_remove = smiles_to_selfies_tokens_ds(train_smiles, token_to_id=token_to_id, 
                                                    seq_len=SEQ_LEN, pad='[PAD]')

train_smiles = np.delete(train_smiles, indieces_to_remove, axis=0)
train_props = np.delete(train_props, indieces_to_remove, axis=0)

# Tokenization smiles with time shift 
decoder_in, train_target = prepare_smiles_to_decoder_ds(train_smiles, token_to_id=token_to_id, 
                                                        seq_len=SEQ_LEN, start='[START]', end='[END]', pad='[PAD]')

# Data scaling
std_sclr = joblib.load('Transformer/StandardScaler.pkl')
train_props = std_sclr.transform(train_props)

Number of skipped smiles: 95
Number of skipped smiles: 0


### **Generator building**

In [ ]:
from Transformer.transformer import TransformerNoEnc
from Gumbel.gumbel import GumbelSoftmax

# Model creation
num_layers = 2
d_model = 64
dff = 128
num_heads = 4
dropout = 0.1
SEQ_LEN = 25
vocab_size = len(token_to_id)

transformer_layer = TransformerNoEnc(
	num_layers=num_layers,
	d_model=d_model,
	num_heads=num_heads,
	dff=dff,
	seq_len=SEQ_LEN,
    vocab_size=vocab_size,
	dropout=dropout
)

# Model buildng for loading weights
props_dim = train_props.shape[-1]
tmp1, tmp2 = np.random.randn(1, props_dim), np.random.randn(1, 1)
tmp = transformer_layer((tmp1, tmp2))

gumbel_softmax_layer = GumbelSoftmax(transformer_layer)
tmp = gumbel_softmax_layer((tmp1, tmp2), beta=1)

/home/ubuntu/miniconda3/envs/new_tensorflow_2/lib/python3.12/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'positional_encoding' (of type PositionalEncoding) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/home/ubuntu/miniconda3/envs/new_tensorflow_2/lib/python3.12/site-packages/keras/src/ops/nn.py:946: UserWarning: You are using a softmax over axis 3 of a tensor of shape (1, 4, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


### **Discriminator building**

In [ ]:
class EmbMatMul(tf.keras.layers.Layer):
	"""Matmul operation"""
	def __init__(self, embedding_layer, **kwargs):
		super().__init__(**kwargs)
		self.embedding_layer = embedding_layer

	def call(self, x):
		return tf.matmul(x, self.embedding_layer.embeddings)
	
def build_discriminator(vocab_size, noise_std, emb_dim, num_filters, filter_sizes, props_dim, dropout, **kwargs):
	"""Discriminator creation"""
	inputs_selfies = tf.keras.layers.Input(shape=(None, vocab_size))
	inputs_props = tf.keras.layers.Input(shape=(props_dim, ))
	
	embeddings = tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=emb_dim, mask_zero=False)
	embeddings.build((None, ))

	x = EmbMatMul(embeddings)(inputs_selfies)
	x = tf.keras.layers.GaussianNoise(noise_std)(x)

	conv_outputs = []
	for filter_size in filter_sizes:
		y = tf.keras.layers.SpectralNormalization(tf.keras.layers.Conv1D(num_filters, filter_size))(x)
		y = tf.keras.layers.ReLU()(y)
		y = tf.keras.layers.GaussianNoise(noise_std)(y)
		y = tf.keras.layers.GlobalMaxPooling1D()(y)
		conv_outputs.append(y)
	
	x1 = tf.keras.layers.Concatenate()(conv_outputs)
	x2 = tf.keras.layers.SpectralNormalization(tf.keras.layers.Dense(64))(inputs_props)
	x2 = tf.keras.layers.ReLU()(x2)

	x = tf.keras.layers.Concatenate()((x1, x2))
	x = tf.keras.layers.Dropout(dropout)(x)

	outputs = tf.keras.layers.SpectralNormalization(tf.keras.layers.Dense(1))(x)

	return tf.keras.Model((inputs_selfies, inputs_props), outputs)

### **Generator & Discriminator building**

In [ ]:
def discriminator_loss(true_samples, gen_logits):
	true_term = tf.nn.relu(1.0 - true_samples)
	gen_term = tf.nn.relu(1.0 + gen_logits)
	return tf.reduce_mean(true_term + gen_term)

def generator_loss(gen_logits):
	return -tf.reduce_mean(gen_logits)


# Generator
gen_input_context = tf.keras.layers.Input((props_dim, ))
gen_input_dec = tf.keras.layers.Input((None, ))
gumbel_softmax_layer_out = gumbel_softmax_layer((gen_input_context, gen_input_dec), beta=1)
generator = tf.keras.Model(inputs=(gen_input_context, gen_input_dec), outputs=gumbel_softmax_layer_out)

# Discriminator
filter_sizes = (2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22)
discriminator = build_discriminator(vocab_size=vocab_size, noise_std=0.01, 
									num_filters=1512, emb_dim=1224, filter_sizes=filter_sizes, props_dim=props_dim, dropout=0.5)

### **GAN's hyperparameters**

In [ ]:
batch_size = 32
steps_per_epoch = len(ds_smiles) // batch_size
epochs = 10

# Learning rate's scheduler
total_steps = steps_per_epoch * epochs
lr_schedule_g = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-5,
    decay_steps=total_steps,
    alpha=1e-7
)

lr_schedule_d = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=total_steps,
    alpha=1e-5
)

gen_optimizer = tf.keras.optimizers.RMSprop(lr_schedule_g)
disc_optimizer = tf.keras.optimizers.RMSprop(lr_schedule_d)

# Weights loading from pratrained transformer
pretrained_transformer_path = 'SavedWeights/pretrain_transformer_weights.weights.h5'
transformer_layer.load_weights(pretrained_transformer_path)

### **Data Packing**

In [8]:
true_selfies = tf.one_hot(train_target, depth=vocab_size)
train_ds = tf.data.Dataset.from_tensor_slices((train_props, decoder_in, true_selfies))
train_ds = train_ds.shuffle(1000).batch(batch_size, drop_remainder=True).prefetch(1)

### **Training**

In [ ]:
def train_gan(generator, discriminator, train_ds, epochs, beta_final, start_token, seq_len):
	"""Main generator training"""
	r1_term = 10.0
	for epoch in range(epochs):
		curr_beta = beta_final ** (epoch / epochs)
        
		for idx, x_batch in enumerate(train_ds):
			targ_props, dec_in_i, targ_selfies = x_batch

			# Discriminator training
			with tf.GradientTape() as d_tape:
				fake_samples = generator((targ_props, dec_in_i), beta=curr_beta, training=True)
				with tf.GradientTape() as r1_d_tape:
					r1_d_tape.watch(targ_selfies)
					real_logits = discriminator((targ_selfies, targ_props), training=True)
				fake_logits = discriminator((fake_samples, targ_props), training=True)
				d_loss = discriminator_loss(true_samples=real_logits, gen_logits=fake_logits)

				# R1
				grad_real = r1_d_tape.gradient(real_logits, targ_selfies)
				r1_pen = tf.reduce_mean(tf.reduce_sum(tf.square(grad_real), axis=[1, 2]))
				d_loss += r1_term * r1_pen
				grads = d_tape.gradient(d_loss, discriminator.trainable_variables)
				disc_optimizer.apply_gradients(zip(grads, discriminator.trainable_variables))
			
			# Generator Training
			if idx % 2 == 0:
				with tf.GradientTape() as g_tape:
					fake_samples = generator((targ_props, dec_in_i), beta=curr_beta, training=True)
					fake_logits = discriminator((fake_samples, targ_props), training=True)
					g_loss = generator_loss(fake_logits)
				grads = g_tape.gradient(g_loss, generator.trainable_variables)
				gen_optimizer.apply_gradients(zip(grads, generator.trainable_variables))
			
			# Showing shample
			if idx % 500 == 0:
				random_sample = targ_props[:1]
				start_id = token_to_id[start_token]
				dec_tokens = [start_id]

				for _ in range(seq_len):
					dec_curr = tf.constant([dec_tokens])
					logits = generator((random_sample, dec_curr), beta=curr_beta)
					next_logits = logits[:, -1, :]
					token = int(tf.argmax(next_logits, axis=-1))
					dec_tokens.append(token)
				print(dec_tokens)

				generator.save_weights('SavedWeights/generator.weights.h5')	
			
			print(f"=== [{idx}] disc. loss: {d_loss:.3f} | gen. loss: {g_loss:.3f} | curr beta: {curr_beta:.3f}")

train_gan(generator, discriminator, train_ds, epochs, beta_final=10, start_token='[START]', seq_len=SEQ_LEN)

2026-05-06 11:40:03.396557: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002


[1, 24, 18, 7, 18, 13, 18, 5, 18, 7, 18, 13, 18, 18, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]
=== [0] disc. loss: 2.385 | gen. loss: 2.590 | curr beta: 1.000
=== [1] disc. loss: 51.750 | gen. loss: 2.590 | curr beta: 1.000
=== [2] disc. loss: 28.254 | gen. loss: 1.188 | curr beta: 1.000
=== [3] disc. loss: 2.305 | gen. loss: 1.188 | curr beta: 1.000
=== [4] disc. loss: 2.377 | gen. loss: 2.676 | curr beta: 1.000
=== [5] disc. loss: 13.064 | gen. loss: 2.676 | curr beta: 1.000
=== [6] disc. loss: 2.178 | gen. loss: -0.205 | curr beta: 1.000
=== [7] disc. loss: 1.841 | gen. loss: -0.205 | curr beta: 1.000
=== [8] disc. loss: 1.654 | gen. loss: 0.137 | curr beta: 1.000
=== [9] disc. loss: 1.656 | gen. loss: 0.137 | curr beta: 1.000
=== [10] disc. loss: 1.520 | gen. loss: 0.424 | curr beta: 1.000
=== [11] disc. loss: 1.865 | gen. loss: 0.424 | curr beta: 1.000
=== [12] disc. loss: 1.861 | gen. loss: 0.414 | curr beta: 1.000
=== [13] disc. loss: 1.884 | gen. loss: 0.414 | curr beta: 1.000
=== [1

2026-05-06 12:19:14.368590: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


[1, 18, 18, 18, 24, 18, 14, 14, 18, 18, 27, 14, 18, 27, 7, 18, 6, 4, 10, 10, 0, 0, 15, 4, 4, 15]
=== [0] disc. loss: 1.752 | gen. loss: 0.551 | curr beta: 1.259
=== [1] disc. loss: 1.727 | gen. loss: 0.551 | curr beta: 1.259
=== [2] disc. loss: 1.798 | gen. loss: 0.562 | curr beta: 1.259
=== [3] disc. loss: 1.723 | gen. loss: 0.562 | curr beta: 1.259
=== [4] disc. loss: 1.744 | gen. loss: 0.603 | curr beta: 1.259
=== [5] disc. loss: 1.776 | gen. loss: 0.603 | curr beta: 1.259
=== [6] disc. loss: 1.755 | gen. loss: 0.504 | curr beta: 1.259
=== [7] disc. loss: 1.750 | gen. loss: 0.504 | curr beta: 1.259
=== [8] disc. loss: 1.737 | gen. loss: 0.547 | curr beta: 1.259
=== [9] disc. loss: 1.723 | gen. loss: 0.547 | curr beta: 1.259
=== [10] disc. loss: 1.745 | gen. loss: 0.584 | curr beta: 1.259
=== [11] disc. loss: 1.793 | gen. loss: 0.584 | curr beta: 1.259
=== [12] disc. loss: 1.793 | gen. loss: 0.592 | curr beta: 1.259
=== [13] disc. loss: 1.791 | gen. loss: 0.592 | curr beta: 1.259
===

2026-05-06 13:38:01.289368: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


[1, 24, 18, 7, 18, 11, 18, 13, 18, 14, 18, 24, 12, 18, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
=== [0] disc. loss: 1.994 | gen. loss: 0.007 | curr beta: 1.995
=== [1] disc. loss: 1.990 | gen. loss: 0.007 | curr beta: 1.995
=== [2] disc. loss: 1.982 | gen. loss: -0.005 | curr beta: 1.995
=== [3] disc. loss: 2.009 | gen. loss: -0.005 | curr beta: 1.995
=== [4] disc. loss: 1.996 | gen. loss: -0.031 | curr beta: 1.995
=== [5] disc. loss: 1.970 | gen. loss: -0.031 | curr beta: 1.995
=== [6] disc. loss: 1.992 | gen. loss: 0.009 | curr beta: 1.995
=== [7] disc. loss: 1.986 | gen. loss: 0.009 | curr beta: 1.995
=== [8] disc. loss: 1.975 | gen. loss: 0.002 | curr beta: 1.995
=== [9] disc. loss: 1.957 | gen. loss: 0.002 | curr beta: 1.995
=== [10] disc. loss: 2.007 | gen. loss: 0.005 | curr beta: 1.995
=== [11] disc. loss: 1.974 | gen. loss: 0.005 | curr beta: 1.995
=== [12] disc. loss: 1.971 | gen. loss: 0.038 | curr beta: 1.995
=== [13] disc. loss: 1.968 | gen. loss: 0.038 | curr beta: 1.995
=== [

2026-05-06 16:17:28.334390: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


[1, 24, 5, 18, 18, 26, 18, 27, 27, 18, 18, 27, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
=== [0] disc. loss: 1.988 | gen. loss: 0.113 | curr beta: 5.012
=== [1] disc. loss: 2.021 | gen. loss: 0.113 | curr beta: 5.012
=== [2] disc. loss: 2.003 | gen. loss: 0.095 | curr beta: 5.012
=== [3] disc. loss: 1.969 | gen. loss: 0.095 | curr beta: 5.012
=== [4] disc. loss: 2.004 | gen. loss: 0.103 | curr beta: 5.012
=== [5] disc. loss: 2.017 | gen. loss: 0.103 | curr beta: 5.012
=== [6] disc. loss: 1.981 | gen. loss: 0.104 | curr beta: 5.012
=== [7] disc. loss: 2.000 | gen. loss: 0.104 | curr beta: 5.012
=== [8] disc. loss: 2.015 | gen. loss: 0.067 | curr beta: 5.012
=== [9] disc. loss: 1.965 | gen. loss: 0.067 | curr beta: 5.012
=== [10] disc. loss: 1.997 | gen. loss: 0.076 | curr beta: 5.012
=== [11] disc. loss: 2.039 | gen. loss: 0.076 | curr beta: 5.012
=== [12] disc. loss: 1.976 | gen. loss: 0.094 | curr beta: 5.012
=== [13] disc. loss: 1.994 | gen. loss: 0.094 | curr beta: 5.012
=== [14] di

In [ ]:
# Generator's weights saving
generator.save_weights('SavedWeights/generator.weights.h5')	